In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from rosbags.highlevel import AnyReader
from rosbags.typesys import Stores, get_types_from_msg, get_typestore

sns.set_theme(style="whitegrid")

REPO = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
BAG_ID = "30618_0d865417"
BAG_PATH = REPO / "data" / BAG_ID
MSG_PATH = REPO / "tram_vehicle_msgs" / "msg"

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
typestore = get_typestore(Stores.ROS2_HUMBLE)

custom_types = {}

custom_types.update(
    get_types_from_msg(
        (MSG_PATH / "VelocitySensor.msg").read_text(encoding="utf-8"),
        "tram_vehicle_msgs/msg/VelocitySensor",
    )
)

custom_types.update(
    get_types_from_msg(
        (MSG_PATH / "DriverControllerCommand.msg").read_text(encoding="utf-8"),
        "tram_vehicle_msgs/msg/DriverControllerCommand",
    )
)

typestore.register(custom_types)

In [ ]:
with AnyReader(
    [BAG_PATH],
    default_typestore=typestore,
) as reader:
    print("Количество сообщений:", reader.message_count)

    topics = pd.DataFrame(
        [
            {
                "topic": connection.topic,
                "type": connection.msgtype,
                "messages": connection.msgcount,
            }
            for connection in reader.connections
        ]
    )

topics

In [ ]:
allowed_topics = {
    "/vehicle/front_bogie_velocity",
    "/vehicle/rear_bogie_velocity",
    "/vehicle/driver_position_cmd",
}

rows = []

with AnyReader(
    [BAG_PATH],
    default_typestore=typestore,
) as reader:
    connections = [
        connection
        for connection in reader.connections
        if connection.topic in allowed_topics
    ]

    for connection, recorded_ns, rawdata in reader.messages(
        connections=connections
    ):
        message = reader.deserialize(rawdata, connection.msgtype)

        header_ns = (
            message.header.stamp.sec * 1_000_000_000
            + message.header.stamp.nanosec
        )

        if connection.topic.endswith("driver_position_cmd"):
            channel = "controller"
            raw_value = int(message.position)
            value = raw_value / 15.0
        elif connection.topic.endswith("front_bogie_velocity"):
            channel = "front"
            raw_value = float(message.velocity)
            value = raw_value
        else:
            channel = "rear"
            raw_value = float(message.velocity)
            value = raw_value

        rows.append(
            {
                "bag_id": BAG_ID,
                "topic": connection.topic,
                "channel": channel,
                "recorded_ns": recorded_ns,
                "header_ns": header_ns,
                "delay_ms": (recorded_ns - header_ns) / 1_000_000,
                "raw_value": raw_value,
                "value": value,
            }
        )

data = pd.DataFrame(rows)

origin_ns = data["header_ns"].min()
data["time_s"] = (data["header_ns"] - origin_ns) / 1e9

data.head()

In [ ]:
data.groupby("channel").agg(
    count=("value", "size"),
    minimum=("value", "min"),
    maximum=("value", "max"),
    mean=("value", "mean"),
    median_delay_ms=("delay_ms", "median"),
    p95_delay_ms=("delay_ms", lambda x: x.quantile(0.95)),
)

In [ ]:
fig, axes = plt.subplots(
    2,
    1,
    figsize=(16, 9),
    sharex=True,
)

for channel in ("front", "rear"):
    part = data[data["channel"] == channel]
    axes[0].plot(
        part["time_s"],
        part["value"],
        label=channel,
        linewidth=1,
    )

controller = data[data["channel"] == "controller"]

axes[1].step(
    controller["time_s"],
    controller["raw_value"],
    where="post",
    label="controller position",
)

axes[0].set_ylabel("Скорость, м/с")
axes[1].set_ylabel("Позиция ручки")
axes[1].set_xlabel("Время от начала, с")

axes[0].legend()
axes[1].legend()
plt.show()

In [ ]:
wheels = (
    data[data["channel"].isin(["front", "rear"])]
    .pivot_table(
        index="header_ns",
        columns="channel",
        values="value",
        aggfunc="last",
    )
    .dropna()
    .reset_index()
)

wheels["time_s"] = (wheels["header_ns"] - origin_ns) / 1e9
wheels["difference_mps"] = wheels["front"] - wheels["rear"]
wheels["median_mps"] = wheels[["front", "rear"]].median(axis=1)

wheels.describe()

In [ ]:
plt.figure(figsize=(16, 4))
plt.plot(
    wheels["time_s"],
    wheels["difference_mps"],
    linewidth=0.8,
)
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("Front − rear, м/с")
plt.xlabel("Время, с")
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

sns.histplot(
    data=data[
        data["delay_ms"].between(-200, 500)
    ],
    x="delay_ms",
    hue="channel",
    bins=150,
    element="step",
    stat="density",
    common_norm=False,
)

plt.xlabel("recorded time − header time, мс")
plt.show()

In [ ]:
output = REPO / "artifacts" / "analysis"
output.mkdir(parents=True, exist_ok=True)

data.to_parquet(
    output / f"{BAG_ID}-inputs.parquet",
    index=False,
)